We consider a gas turbine generator with a temperature-dependent quadratic cost function

$$ C(x) = (1+ \gamma \tau) (\frac{\alpha}{2} x^2 +  \beta x), $$

where $\tau$ is the ambient temperature in Kelvin.

The temperature-dependent factor accounts for the fact that the thermal efficiency of
a gas turbine generator decreases with respect to ambient temperature.

Under such cost function, the gas turbine supplier respond to a price $p$ with quantity

$$ x^* = \frac{p - (1 + \gamma \tau) \beta_0}{(1 + \gamma \tau)\alpha}. $$

In [1]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from numba import jit, njit

import json

In [2]:
@njit
def supply_from_price(p, a, b):
    return (p - b) / a

@njit
def cost_func(x, a, b):
    return a * x**2 / 2 + b * x

@njit
def optimal_price(x, a, b):
    return a * x + b

In [3]:
# Problem setup
a, b = np.meshgrid(np.linspace(1, 1.2, 5), np.linspace(0, 0.2, 5))
a, b = a.ravel(), b.ravel()

T_range = 2**np.arange(4,24,1)
num_instances = 10

Implementing online regression oracle with multiplicative weights algorithm

Note: using the averaged variant in Ch.3 of [5]

[5] Cesa-Bianchi, Nicolo, and Gábor Lugosi. Prediction, learning, and games. Cambridge university press, 2006.

In [4]:
class OnlineReg:
    def __init__(self, a, b, eta=1/2):
        self.a, self.b = a, b
        self.eta = eta

        self.n = len(a)
        self.q = np.zeros_like(a)

    def update(self, p, c, y):
        self.q += self.eta * (supply_from_price(p, c*self.a, c*self.b) - y)**2
        # shift the weights to ensure numerical stability
        self.q -= self.q.min()

    def pred(self, p, c):
        w = np.exp(-self.q) / np.sum(np.exp(-self.q))
        return np.dot(supply_from_price(p, c*self.a, c*self.b), w)

In [5]:
@njit
def emw_update(q, p, a, b, y, eta=1/2):
    q += eta * (supply_from_price(p, a, b) - y)**2
    # shift the weights to ensure numerical stability
    q -= q.min()
    
@njit
def emw_pred(q, p, a, b):
    w = np.exp(-q) / np.sum(np.exp(-q))
    return np.dot(supply_from_price(p, a, b), w)

Implementing Algoirthm 4

In [6]:
@njit
def normalize(weights, K, eps=1e-5):
    lo, hi = 1, K
    s = 0.

    while np.abs(s - 1.) > eps:
        mi = (lo + hi) / 2
        s = np.sum(1 / (mi + weights))

        if s > 1:
            lo = mi
        else:
            hi = mi

    probs = 1 / (mi + weights)
    return probs / np.sum(probs)

@njit
def numba_weighted_choice(arr, p):
    """Equivalent to np.random.choice(arr, p=p) for a 1D array"""
    cdf = np.cumsum(p)
    cdf /= cdf[-1] 
    
    r = np.random.random()
    idx = np.searchsorted(cdf, r)
    
    return arr[idx]

@njit
def varying_cost_alg(demand, ctx, a, b, true_a, true_b):
    T = len(demand)
    n = len(a)

    K = int((T)**(1/3))
    ex = np.sqrt(T * K / np.log(n))
    discrete_prices = np.linspace(0, 1, K)

    # oracle weights
    q = np.zeros_like(a)

    unmet_demand = 0.0
    cost_regret = 0.0
    payment_regret = 0.0

    for t, (d, c) in enumerate(zip(demand, ctx)):
        diff = np.abs(emw_pred(q, discrete_prices[:, np.newaxis], a*c, b*c) - d)
        weights = normalize(2 * ex * (diff - diff.min()), K)
        
        phat = numba_weighted_choice(discrete_prices, weights)

        supplied = supply_from_price(phat, c*true_a, c*true_b)
        pstar = optimal_price(d, c*true_a, c*true_b)

        unmet_demand += max(d - supplied, 0.0)
        cost_regret += cost_func(supplied, c*true_a, c*true_b) - cost_func(d, c*true_a, c*true_b)
        payment_regret += phat * supplied - pstar * d

        emw_update(q, phat, a*c, b*c, supplied)

    return unmet_demand, cost_regret, payment_regret

In [7]:
def mean_and_ci(arr):
    mean = arr.mean(axis=0)
    sd = arr.std(axis=0, ddof=1)
    se = sd / np.sqrt(arr.shape[0])
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    return mean, ci_low, ci_high

In [ ]:
%%time
max_T = max(T_range)
rng = np.random.default_rng(0)

unmet_demand_regret_trials = np.zeros((num_instances, len(T_range)))
cost_regret_trials = np.zeros((num_instances, len(T_range)))
payment_regret_trials = np.zeros((num_instances, len(T_range)))

for trial in range(num_instances):
    raw_noise = rng.normal(0., 1., size=max_T)
    smooth_noise = np.convolve(raw_noise, np.array([0.2, 0.6, 0.2]), mode="same")

    demand = 0.5 + np.clip(smooth_noise * 0.02, -0.05, 0.05)
    
    ctx = 1.05 + 0.05 * np.sin(0.1 * np.arange(max_T))

    idx = rng.integers(len(a))
    true_a, true_b = a[idx], b[idx]

    for j, T in enumerate(T_range):
        oracle = OnlineReg(a, b)
    
        unmet_demand, cost_regret, payment_regret = varying_cost_alg(demand[:T], ctx[:T], a, b, true_a, true_b)

        unmet_demand_regret_trials[trial, j] = unmet_demand
        cost_regret_trials[trial, j] = cost_regret
        payment_regret_trials[trial, j] = payment_regret

In [ ]:
np.savez('varying_cost.npz',
         T_range = T_range,
         unmet_demand_regret_trials = unmet_demand_regret_trials,
         cost_regret_trials = cost_regret_trials,
         payment_regret_trials = payment_regret_trials,
         allow_pickle = False,
)

In [ ]:
res = {}
for name, arr in np.load('varying_cost.npz').items():
    res[name] = arr

locals().update(res)

In [ ]:
figsize = (6, 4)
postfix = ''
plt.rcParams.update({'font.size': 11})

In [ ]:
def mean_and_ci(arr):
    mean = arr.mean(axis=0)
    sd = arr.std(axis=0, ddof=1)
    se = sd / np.sqrt(arr.shape[0])
    ci_low = mean - 1.96 * se
    ci_high = mean + 1.96 * se
    return mean, ci_low, ci_high

In [ ]:
logT = np.log(np.array(T_range))

mid_idx = len(logT) // 2
anchor_T = logT[mid_idx]

# unmet demand
unmet_demand_regret_arr_full, unmet_ci_low, unmet_ci_high = mean_and_ci(unmet_demand_regret_trials)
logR = np.log(np.maximum(unmet_demand_regret_arr_full, 1e-12))
log_ci_low = np.log(np.maximum(unmet_ci_low, 1e-12))
log_ci_high = np.log(np.maximum(unmet_ci_high, 1e-12))
anchor_R = logR[mid_idx]
ref_line = 2/3 * (logT - anchor_T) + anchor_R

plt.figure(figsize=figsize)
plt.plot(logT, logR, marker='o', markersize=5, linewidth=3, label='Algorithm 4')
plt.fill_between(logT, log_ci_low, log_ci_high, alpha=0.25, label='95% CI')
plt.plot(logT, ref_line, '--', linewidth=3, label='Slope = 2/3')
plt.xlabel('log(T)')
plt.ylabel('log(Unmet demand)')
#plt.title(f'{plot_name}: unmet demand')
plt.legend()
plt.savefig(f'varying_cost_unmet_demand_loglog{postfix}.pdf', bbox_inches='tight')
plt.show()

# cost regret
cost_abs_trials = np.abs(cost_regret_trials)
cost_abs_mean, cost_abs_ci_low, cost_abs_ci_high = mean_and_ci(cost_abs_trials)
logR = np.log(np.maximum(cost_abs_mean, 1e-12))
log_ci_low = np.log(np.maximum(cost_abs_ci_low, 1e-12))
log_ci_high = np.log(np.maximum(cost_abs_ci_high, 1e-12))
anchor_R = logR[mid_idx]
ref_line = 2/3 * (logT - anchor_T) + anchor_R

plt.figure(figsize=figsize)
plt.plot(logT, logR, marker='o', markersize=5, linewidth=3, label='Algorithm 4')
plt.fill_between(logT, log_ci_low, log_ci_high, alpha=0.25, label='95% CI')
plt.plot(logT, ref_line, '--', linewidth=3, label='Slope = 2/3')
plt.xlabel('log(T)')
plt.ylabel('log(|Cost Regret|)')
#plt.legend()
#plt.title(f'{plot_name}: cost regret')
plt.savefig(f'varying_cost_cost_regret_loglog{postfix}.pdf', bbox_inches='tight')
plt.show()

# Payment regret
payment_abs_trials = np.abs(payment_regret_trials)
payment_abs_mean, payment_abs_ci_low, payment_abs_ci_high = mean_and_ci(payment_abs_trials)
logR = np.log(np.maximum(payment_abs_mean, 1e-12))
log_ci_low = np.log(np.maximum(payment_abs_ci_low, 1e-12))
log_ci_high = np.log(np.maximum(payment_abs_ci_high, 1e-12))
anchor_R = logR[mid_idx]
ref_line = 2/3 * (logT - anchor_T) + anchor_R

plt.figure(figsize=figsize)
plt.plot(logT, logR, marker='o', markersize=5, linewidth=3, label='Algorithm 4')
plt.fill_between(logT, log_ci_low, log_ci_high, alpha=0.25, label='95% CI')
plt.plot(logT, ref_line, '--', linewidth=3, label='Slope = 2/3')
plt.xlabel('log(T)')
plt.ylabel('log(|Payment Regret|)')
#plt.legend()
#plt.title(f'{plot_name}: payment regret')
plt.savefig(f'varying_cost_payment_regret_loglog{postfix}.pdf', bbox_inches='tight')
plt.show()

In [ ]:
unmet_slope = stats.linregress(np.log(T_range), np.log(np.maximum(unmet_demand_regret_arr_full, 1e-12))).slope
cost_slope = stats.linregress(np.log(T_range), np.log(np.maximum(np.abs(cost_abs_mean), 1e-12))).slope
payment_slope = stats.linregress(np.log(T_range), np.log(np.maximum(np.abs(payment_abs_mean), 1e-12))).slope

print('Unmet demand regret slope:', unmet_slope)
print('cost regret slope:', cost_slope)
print('Payment regret slope:', payment_slope)